In [ ]:
import numpy as np

#executar uv sync

#1. Base : [tempo de uso (h), volume_transacoes (k) e classe (0: Básico, 1: Premium)]
X_treino = np.array([
    [1.0, 1.0],
    [2.0, 2.0],
    [3.0, 1.0],
    [6.0, 5.0],
    [7.0, 6.0],
    [8.0, 7.0]
])

y_treino = np.array([0, 0, 0, 1, 1, 1])

#2. Novo ponto que queremos classificar
novo_ponto = np.array([6.5, 5.5])

#3. Distâncias euclidianas vetorizadas de uma só vez para todas as linhas
#np.linalg.norm sqrt((x1 - p1)^2 + (x2 - p2)^2) para cada linha
#ord=2 calcula a distancia euclidiana e ord=1 calcula a distancia manhantan
dist_euclidiana = np.linalg.norm(X_treino - novo_ponto, ord=2, axis = 1) #a diferença de um ponto para o outro preciso da diferença para elevar ao quadrado e encontrarmos a hipotenusa
dist_manhatan= np.linalg.norm(X_treino - novo_ponto, ord=1, axis = 1) #a diferença de um ponto para o outro preciso da diferença para elevar ao quadrado e encontrarmos a hipotenusa

#4. Encontra os índices dos K vizinhos mais próximos ( k = 3)
#np.argsort retorna os índices que ordenam o vetor da menor para a maior distância
k = 3
vizinhos_idx = np.argsort(dist_euclidiana)[:k] #trocar manualmente entre manhatan e euclidiana

#5. Votacao da maioria (moda dos rótulos dos vizinhos)
# Y_treino[vizinhos_idx] obtem as classes dos 3 vizinhos (ex: [1, 1, 0])
#np.bincount conta a frequencia de cada classe
# .argmax() retorna a classe com a maior quantidade de votos
votos = y_treino[vizinhos_idx]
classe_predita = int(np.bincount(votos).argmax())

print(f"Distâncias euclidianas para cada amostra: {np.round(dist_euclidiana, 2)}")
print(f"Distâncias manhatan para cada amostra: {np.round(dist_manhatan, 2)}")
print(f"Índices dos {k} vizinhos: {vizinhos_idx}")
print(f"Classes dos vizinhos: {votos}")
print(f"Classe predita: {'Premium (1)' if classe_predita == 1 else 'Básico (0)'}")

Distâncias para cada amostra: [7.11 5.7  5.7  0.71 0.71 2.12]
Índices dos 3 vizinhos: [3 4 5]
Classes dos vizinhos: [1 1 1]
Classe predita: Premium (1)


In [ ]:
def calcular_distancias(X, ponto, metrica="euclidiana"):
    """calcula distancias vetorizadas para todas as linhas da matriz X.
    A Euclidiana é a ord=2 e a manhatan é a ord=1"""

    if metrica == "euclidiana":
        return np.linalg.norm( X - ponto, ord=2, axis=1)
    elif metrica == "manhatan":
        return np.linalg.norm( X - ponto, ord=1, axis=1)
    else:
        raise ValueError("Metrica invalida. Escolha'euclidiana' ou 'manhanta'.")

def knn_classificar(X, y, ponto, k=3, metrica="euclidiana"):
    """Classifica um ponto com base na votação dos k vizinhos mais próximos"""

    #1. Calcula distâncias vetorizadas conforme a métrica escolhida
    distancias = calcular_distancias(X, ponto, metrica=metrica)

    #2. Localiza os k índices mais próximos
    vizinhos = np.argsort(distancias)[:k]

    #3. Elege a classe majoritaria (moda)
    classe_vencedora = int(np.bincount(y[vizinhos]).argmax())

    return classe_vencedora, vizinhos, distancias[vizinhos]

def knn_recomendar(X, ponto_referencia, k=3, metrica="euclidiana"):
    """Retorna os indices mais distancias dos k itens mais semelhantes do catalogo."""

    #1. Calcula distâncias vetorizadas conforme a métrica escolhida
    distancias = calcular_distancias(X, ponto_referencia, metrica=metrica)

    #2. Retorna os k itens mais próximos (menor distancia)
    vizinhos = np.argsort(distancias)[:k]

    return vizinhos, distancias[vizinhos]

In [ ]:
#Teste de classificação
ponto_teste = np.array([2.5, 2.0])

for metrica in ["euclidiana", "manhatan"]:
    classe, vizinhos, dists = knn_classificar(
        X_treino, y_treino, ponto_teste, k=3, metrica=metrica
    )
    rotulo = "Premium" if classe == 1 else "Bifasico"

    print(f"---Métrica: Distancia {metrica.capitalize()} (K=3)")
    print(f"Ponto {ponto_teste} -> Classificado como: {rotulo}")
    print(f"Vizinhos consultados: índices {vizinhos} com distancias {np.round(dists,2)}\n")

In [ ]:
#Recomendação por similaridade
item_referencia = np.array([7.0, 6.5])

for metrica in ["euclidiana", "manhatan"]:
    indices_rec, distancias_rec = knn_recomendar(
        X_treino, item_referencia, k=2, metrica=metrica #para alterar os valores de distância basta alterar o k
    )

    print(f"Top 2 Itens mais parecidos ({metrica.capitalize()}):")
    for pos, (idx, d) in enumerate(zip(indices_rec, distancias_rec), start=1):
        print(f" {pos}° lugar: item #{idx} (Atributos: {X_treino[idx]}) - Distancias: {d:.2f}")
    print()



In [ ]:
#Gráfico mostrando os pontos

import matplotlib.pyplot as plt

#1. Plot das amostras históricas
plt.figure(figsize=(7,5))
plt.scatter(X_treino[y_treino == 0, 0], X_treino[y_treino == 0, 1 ], color="royalblue",
            s=80, label="Básico (0)")
plt.scatter(X_treino[y_treino == 1,0], X_treino[y_treino == 1, 1], color="crimson",
            s=80, label="Premium (1)")

#2. Plot do novo ponto
plt.scatter( novo_ponto[0], novo_ponto[1], color="gold", marker="*", s=200, edgecolors="black",
            label="Novo ponto", norder=5)

#3. Linhas conectando o ponto aos 3 vizinhos mais proximos
for idx in vizinhos_idx:
    plt.plot([novo_ponto[0], X_treino[idx,0]], [novo_ponto[1], X_treino[idx, 1]],
             color="gray", linestyle="--", linewidth=1.2)

plt.title("Classificação por vizinhança (K = 3)")
plt.xlabel("Tempo de uso (h)")
plt.ylabel("Volume financeiro (R$ mil)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.slow()